<a href="https://colab.research.google.com/github/Elchegue64/Tarea-3/blob/main/Tarea_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q tensorflow wandb


In [2]:
import wandb
wandb.login()


Si no tienes API key, crea cuenta en https://wandb.ai/ y copia tu API key


/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize
wandb: Paste an API key from your profile and hit enter:

 ··········


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: terceraluna766 (terceraluna766-buap) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [3]:
import time
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers
import matplotlib.pyplot as plt
import os
import json


In [4]:
# Cargar MNIST desde Keras
(x_train, y_train), (x_test, y_test) = keras.datasets.mnist.load_data()

# Normalizar y aplanar
x_train = x_train.reshape((-1, 784)).astype("float32") / 255.0
x_test  = x_test.reshape((-1, 784)).astype("float32") / 255.0

# One-hot y vector binary_crossentroppy
y_train_cat = keras.utils.to_categorical(y_train, 10)
y_test_cat  = keras.utils.to_categorical(y_test, 10)

print(x_train.shape, y_train_cat.shape)


11490434/11490434 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step
(60000, 784) (60000, 10)


In [5]:
def build_dense_model(layers_sizes, activation_hidden='sigmoid', activation_out='sigmoid',
                      loss='binary_crossentropy', optimizer=None, l1=0.0, l2=0.0, dropout=0.0):

    model = keras.Sequential()
    # capa entrada -> primer hidden
    input_dim = layers_sizes[0]
    # asumimos layers_sizes incluye output; iteramos intermedias
    for i, size in enumerate(layers_sizes[1:]):
        is_last = (i == len(layers_sizes[1:]) - 1)
        if is_last:
            act = activation_out
            # salida
            if dropout > 0:
                model.add(layers.Dropout(dropout))
            model.add(layers.Dense(size, activation=act,
                                   kernel_regularizer=regularizers.L1L2(l1=l1, l2=l2)))
        else:
            # hidden layer
            act = activation_hidden
            if i == 0:
                # primera hidden necesita input_dim
                model.add(layers.Dense(size, activation=act, input_shape=(input_dim,),
                                       kernel_regularizer=regularizers.L1L2(l1=l1, l2=l2)))
            else:
                model.add(layers.Dense(size, activation=act,
                                       kernel_regularizer=regularizers.L1L2(l1=l1, l2=l2)))
            if dropout > 0:
                model.add(layers.Dropout(dropout))
    model.compile(optimizer=optimizer,
                  loss=loss,
                  metrics=['accuracy'])
    return model

def train_and_time(model, x_train, y_train, x_val, y_val, epochs=10, batch_size=128, project_name="exp"):
    """
    Entrena el modelo, Devuelve history y tiempos.
    """
    start_all = time.time()
    times = []
    per_epoch_start = None
    # Callback para medir tiempo por época
    class TimeCallback(keras.callbacks.Callback):
        def on_epoch_begin(self, epoch, logs=None):
            self.epoch_t0 = time.time()
        def on_epoch_end(self, epoch, logs=None):
            t = time.time() - self.epoch_t0
            times.append(t)
    history = model.fit(x_train, y_train, validation_data=(x_val, y_val),
                        epochs=epochs, batch_size=batch_size, verbose=2,
                        callbacks=[TimeCallback()])
    total = time.time() - start_all
    return history, times, total


In [6]:
# Parámetros comunes
epochs = 10
batch_size = 32

baseline_optimizer = keras.optimizers.SGD(learning_rate=3.0)
baseline_model = build_dense_model([784, 30, 10],
                                   activation_hidden='sigmoid',
                                   activation_out='sigmoid',
                                   loss='binary_crossentropy',
                                   optimizer=baseline_optimizer,
                                   l1=0.0, l2=0.0, dropout=0.0)

# Entrenar baseline
history_base, times_base, total_base = train_and_time(baseline_model,
                                                      x_train, y_train_cat,
                                                      x_test,  y_test_cat,
                                                      epochs=epochs, batch_size=batch_size)
print("Tiempos por época baseline:", times_base)
print("Total (s):", total_base)


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/10
1875/1875 - 6s - 3ms/step - accuracy: 0.8821 - loss: 0.0796 - val_accuracy: 0.9288 - val_loss: 0.0487
Epoch 2/10
1875/1875 - 4s - 2ms/step - accuracy: 0.9325 - loss: 0.0453 - val_accuracy: 0.9409 - val_loss: 0.0404
Epoch 3/10
1875/1875 - 5s - 3ms/step - accuracy: 0.9440 - loss: 0.0385 - val_accuracy: 0.9489 - val_loss: 0.0362
Epoch 4/10
1875/1875 - 5s - 3ms/step - accuracy: 0.9506 - loss: 0.0346 - val_accuracy: 0.9508 - val_loss: 0.0336
Epoch 5/10
1875/1875 - 6s - 3ms/step - accuracy: 0.9557 - loss: 0.0316 - val_accuracy: 0.9545 - val_loss: 0.0319
Epoch 6/10
1875/1875 - 4s - 2ms/step - accuracy: 0.9587 - loss: 0.0295 - val_accuracy: 0.9547 - val_loss: 0.0314
Epoch 7/10
1875/1875 - 4s - 2ms/step - accuracy: 0.9610 - loss: 0.0278 - val_accuracy: 0.9570 - val_loss: 0.0302
Epoch 8/10
1875/1875 - 6s - 3ms/step - accuracy: 0.9636 - loss: 0.0265 - val_accuracy: 0.9589 - val_loss: 0.0292
Epoch 9/10
1875/1875 - 5s - 2ms/step - accuracy: 0.9655 - loss: 0.0253 - val_accuracy: 0.9588 - 

In [7]:

wandb.init(project="tarea3_experimentos", reinit=True)

results = {}

# Exeprimento 1: más neuronas (784->64->10), misma activación y SGD (sigmoid)
opt1 = keras.optimizers.SGD(learning_rate=3.0)
m1 = build_dense_model([784, 64, 10], activation_hidden='sigmoid', activation_out='sigmoid', loss='binary_crossentropy', optimizer=opt1)
h1, t1, tot1 = train_and_time(m1, x_train, y_train_cat, x_test, y_test_cat, epochs=epochs, batch_size=batch_size)
results['exp1'] = {'history': h1.history, 'times': t1, 'total': tot1}

# Experimento 2: más profundidad + relu + softmax + Adam (estándar moderno)
opt2 = keras.optimizers.Adam(learning_rate=0.001)
m2 = build_dense_model([784, 64, 32, 10], activation_hidden='relu', activation_out='softmax', loss='categorical_crossentropy', optimizer=opt2)
h2, t2, tot2 = train_and_time(m2, x_train, y_train_cat, x_test, y_test_cat, epochs=epochs, batch_size=batch_size)
results['exp2'] = {'history': h2.history, 'times': t2, 'total': tot2}

# Experimento 3: más ancho + relu + RMSprop
opt3 = keras.optimizers.RMSprop(learning_rate=0.001)
m3 = build_dense_model([784, 128, 64, 10], activation_hidden='relu', activation_out='softmax', loss='categorical_crossentropy', optimizer=opt3)
h3, t3, tot3 = train_and_time(m3, x_train, y_train_cat, x_test, y_test_cat, epochs=epochs, batch_size=batch_size)
results['exp3'] = {'history': h3.history, 'times': t3, 'total': tot3}

# Impresión resumen simple
for k,v in results.items():
    print(k, "val_accuracy last epoch:", v['history']['val_accuracy'][-1], "total_sec:", v['total'])


wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.


Epoch 1/10
1875/1875 - 7s - 4ms/step - accuracy: 0.8869 - loss: 0.0754 - val_accuracy: 0.9326 - val_loss: 0.0464
Epoch 2/10
1875/1875 - 6s - 3ms/step - accuracy: 0.9384 - loss: 0.0418 - val_accuracy: 0.9465 - val_loss: 0.0368
Epoch 3/10
1875/1875 - 7s - 4ms/step - accuracy: 0.9534 - loss: 0.0330 - val_accuracy: 0.9551 - val_loss: 0.0307
Epoch 4/10
1875/1875 - 5s - 3ms/step - accuracy: 0.9607 - loss: 0.0279 - val_accuracy: 0.9604 - val_loss: 0.0279
Epoch 5/10
1875/1875 - 6s - 3ms/step - accuracy: 0.9657 - loss: 0.0247 - val_accuracy: 0.9648 - val_loss: 0.0256
Epoch 6/10
1875/1875 - 6s - 3ms/step - accuracy: 0.9705 - loss: 0.0221 - val_accuracy: 0.9657 - val_loss: 0.0238
Epoch 7/10
1875/1875 - 6s - 3ms/step - accuracy: 0.9733 - loss: 0.0203 - val_accuracy: 0.9682 - val_loss: 0.0219
Epoch 8/10
1875/1875 - 6s - 3ms/step - accuracy: 0.9761 - loss: 0.0187 - val_accuracy: 0.9677 - val_loss: 0.0228
Epoch 9/10
1875/1875 - 5s - 3ms/step - accuracy: 0.9779 - loss: 0.0174 - val_accuracy: 0.9701 - 

In [ ]:
best_arch = [784, 64, 32, 10]
base_optimizer = keras.optimizers.Adam(learning_rate=0.001)

regs = {
    'L1': {'l1':1e-4, 'l2':0, 'dropout':0.0},
    'L2': {'l1':0, 'l2':1e-4, 'dropout':0.0},
    'L1L2': {'l1':1e-4, 'l2':1e-4, 'dropout':0.0},
    'Dropout': {'l1':0, 'l2':0, 'dropout':0.3},
    'Dropout+L1L2': {'l1':1e-4, 'l2':1e-4, 'dropout':0.3}
}

reg_results = {}
for name, params in regs.items():
    print("Entrenando variante:", name)
    model_reg = build_dense_model(best_arch, activation_hidden='relu', activation_out='softmax',
                                  loss='categorical_crossentropy', optimizer=base_optimizer,
                                  l1=params['l1'], l2=params['l2'], dropout=params['dropout'])
    h, t, tot = train_and_time(model_reg, x_train, y_train_cat, x_test, y_test_cat, epochs=epochs, batch_size=batch_size)
    reg_results[name] = {'history': h.history, 'times': t, 'total': tot}
    # guardar historial a disco
    fname = f"history_{name}.json"
    with open(fname, "w") as f:
        json.dump(h.history, f)
    # plot accuracy
    plt.figure()
    plt.plot(h.history['accuracy'], label='train_acc')
    plt.plot(h.history['val_accuracy'], label='val_acc')
    plt.title(f"Acc {name}")
    plt.legend()
    plt.savefig(f"acc_{name}.png")
    plt.close()
    print(name, "val_acc last:", h.history['val_accuracy'][-1], "total_s:", tot)
